# Arabic Poetry Era Classification

---

## 1. Project Overview

This project focuses on classifying Arabic poems into different historical eras using Natural Language Processing (NLP) and Machine Learning techniques. The goal is to build a model that can predict the era of a given Arabic poem based on its content.

## 2. Problem Statement

Given an Arabic poem, can we accurately classify which historical era it belongs to? This is a multi-class text classification problem where we need to distinguish between multiple historical eras of Arabic poetry.

## 3. Dataset Description

The dataset contains Arabic poems with the following columns:
- `poet_name`: Name of the poet
- `poet_era`: Historical era of the poet (target variable)
- `poem_tags`: Tags associated with the poem
- `poem_title`: Title of the poem
- `poem_text`: The actual poem text (feature variable)
- `poem_count`: Number of lines in the poem

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from preprocessing import ArabicTextPreprocessor
from utils import save_object, load_object
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
data = pd.read_csv('Arabic_Poetry_Dataset.csv')
print(f"Dataset shape: {data.shape}")
data.head()

In [ ]:
data.info()

In [ ]:
print("Missing values:")
data.isnull().sum()

In [ ]:
print("Poem eras distribution:")
data['poet_era'].value_counts()

## 4. Data Preprocessing

In [ ]:
data = data.dropna()
print(f"Dataset shape after dropping NA: {data.shape}")

In [ ]:
preprocessor = ArabicTextPreprocessor()
data['cleaned_poem_text'] = data['poem_text'].apply(preprocessor.clean_text)

In [ ]:
print("Before preprocessing:")
print(data['poem_text'].iloc[0][:200])
print("\nAfter preprocessing:")
print(data['cleaned_poem_text'].iloc[0][:200])

## 5. Feature Engineering

In [ ]:
tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 3), analyzer='char_wb')
X = tfidf.fit_transform(data['cleaned_poem_text'])
print(f"TF-IDF matrix shape: {X.shape}")

In [ ]:
encoder = LabelEncoder()
y = encoder.fit_transform(data['poet_era'])
print(f"Classes: {encoder.classes_}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

## 6. Model Training

In [ ]:
models = {
    'Naive Bayes': MultinomialNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
}

trained_models = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"{name} trained successfully!")

## 7. Model Evaluation

In [ ]:
results = []
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results.append({'Model': name, 'Accuracy': accuracy})
    print(f"\n=== {name} ===")
    print(f"Accuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=encoder.classes_))

In [ ]:
results_df = pd.DataFrame(results)
plt.figure(figsize=(10, 6))
sns.barplot(x='Model', y='Accuracy', data=results_df)
plt.title('Model Comparison - Accuracy')
plt.ylim(0, 1)
plt.show()

In [ ]:
best_model_name = results_df.loc[results_df['Accuracy'].idxmax(), 'Model']
best_model = trained_models[best_model_name]
y_pred_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45)
plt.show()

## 8. Results

The best performing model is **{best_model_name}** with an accuracy of **{results_df['Accuracy'].max():.4f}**.

## 9. Conclusion

We successfully built a multi-class classification model to predict the historical era of Arabic poems. The model was trained on a cleaned and preprocessed dataset using TF-IDF for feature extraction, and multiple algorithms were compared for performance.

## 10. Future Improvements

- Use more advanced NLP techniques (e.g., BERT, AraBERT)
- Collect more data for underrepresented classes
- Experiment with different feature extraction methods
- Perform hyperparameter tuning
- Add more visualization of the results

In [ ]:
save_object(best_model, 'model.pkl')
save_object(tfidf, 'tfidf_vectorizer.pkl')
save_object(encoder, 'label_encoder.pkl')
print("Model and preprocessing objects saved successfully!")